# UABR-SLM · Notebook 07 — Phase 5 SFT-only pilot (E0 frozen vs E1 plain SFT) → GO / RECALIBRATION_ONLY / NO_CHANGE

**Question** (uabr-slm-main `docs/pilot_decision.md`, rule fixed before any result): does ordinary instruction tuning
with **no replay** damage the `<ANSWER>`/`<CALL>` policy of θ_route,preSFT in a way that re-choosing the call
threshold **cannot** repair? GPU **T4** (×1) · Internet **On** (Dolly / Alpaca download) · ~45–70 min.

**Setup (protocol unchanged from uabr-slm-main, `configs/sft_methods/standard.yaml`)**

| Item | Value |
|---|---|
| Start | θ_route,preSFT = `06_routing/router_action.pt` (E0 = this model, frozen; reference, not an arm — A2) |
| SFT data | Dolly-15k + Alpaca-cleaned, **plain format** `<USER> instruction<ASSISTANT> response<EOS>` (no action tokens, A5 primary), ≤ 512 tokens, ~6 M tokens, 1 epoch, 500 held-out instruction examples; loss on the response only |
| Optimiser | AdamW (0.9, 0.95), wd 0.01, LR 5e-5 constant after 30 warm-up steps, batch 32 sequences, grad clip 1.0, fp16; all weights train |
| Seeds | 11 and 29 (the seed fixes the instruction batch order) |
| Durations | 25 %, 50 %, 100 % of the updates from one run per seed (constant LR ⇒ equal to shorter runs) |
| Evaluation | All labelled **validation** questions (in_main; test stays sealed); θ_route,preSFT is the 0 % point |
| Re-use | The 100 % checkpoints are the E1 (SFT-only) arm of Phase 6 |

**Measures at 0 / 25 / 50 / 100 %:** instruction validation loss · routing AUROC / AUPRC against the **frozen θ0 labels**
(A3a policy retention) · accuracy and utility at a re-picked 5 / 10 / 20 % call budget with **fresh** rewards (greedy
closed-book and evidence EM of that checkpoint, A3b) · accuracy at the **frozen** 5 / 10 / 20 % thresholds of the 0 % point
· default call rate (z_CALL > z_ANSWER) · harmful-answer rate · closed-book / evidence EM · A4 flips by pre-SFT margin
· A5 action-slot collapse (free-generation emission, P(<ANSWER>)+P(<CALL>), two-way AUROC, re-tuned 10 % threshold).

**Decision rule** (100 % minus 0 %, same validation questions, paired **subject-clustered** bootstrap, 1,000 resamples,
95 % CI; a drop is significant when the upper bound < 0):
- **GO:** in **both** seeds, routing AUROC **or** accuracy at the re-picked 10 % budget drops significantly → Phase 6 is the main claim.
- **RECALIBRATION_ONLY:** not GO, but in both seeds there is damage and at least the frozen-threshold accuracy drops.
- **NO_CHANGE:** otherwise (check that SFT learned and the 0 % policy is not collapsed).

*Adaptations to this repo (declared):* the bootstrap resamples subject clusters (A9) instead of single prompts; the dev
set is our validation split; SFT examples that state a validation/test PopQA fact (subject **and** gold alias in the
text) are removed before the token budget is filled (plan: "audit overlap with PopQA test facts").

**Inputs:** outputs of notebooks **00, 01, 02, 06 (v2)** (optional: a previous run of this notebook — its SFT checkpoints are reused).

## 1 · Get the project code

In [ ]:
import os, sys, glob, json, subprocess, platform, time, shutil
ON_KAGGLE = os.path.exists("/kaggle/working")

REPO_URL = "https://github.com/Prajwal-13-dev/uabr_slm.git"
REPO_REF = "main"        # pin to a commit hash for final, reportable runs

def find_code_root():
    if os.environ.get("UABR_CODE"):
        return os.path.abspath(os.environ["UABR_CODE"])
    for base in glob.glob("/kaggle/input/*"):
        for root, dirs, _ in os.walk(base):
            if os.path.isdir(os.path.join(root, "src", "uabr")):
                return root
            if root.count(os.sep) - base.count(os.sep) >= 4:
                dirs[:] = []
    dest = "/tmp/uabr_slm"
    if not os.path.isdir(dest):
        url = REPO_URL
        try:
            from kaggle_secrets import UserSecretsClient
            tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
            url = url.replace("https://", f"https://{tok}@")
        except Exception:
            pass
        subprocess.run(["git", "clone", "-q", url, dest], check=True)
        subprocess.run(["git", "-C", dest, "checkout", "-q", REPO_REF], check=True)
    return dest

CODE_ROOT = find_code_root()
SRC = os.path.join(CODE_ROOT, "src")
sys.path.insert(0, SRC)
CODE_COMMIT = subprocess.run(["git", "-C", CODE_ROOT, "rev-parse", "HEAD"],
                             capture_output=True, text=True).stdout.strip() or "unknown"

def find_input(pattern, env=None):
    """Locate a file from an attached notebook output (or an env override for local tests)."""
    if env and os.environ.get(env):
        return os.environ[env]
    hits = sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    assert hits, f"{pattern} not found under /kaggle/input - attach the required notebook output (Add Input → Your Work)"
    return hits[0]

T0 = time.time()
def stage(msg): print(f"[{(time.time()-T0)/60:6.1f} min] {msg}", flush=True)
print("code:", CODE_ROOT, "| commit:", CODE_COMMIT[:10])

import torch
from uabr.data import tokenizer as T
from uabr.data import qa_format as Q
from uabr.data.sft_data import SOURCES, rows_from_hf, build_sft, audit_overlap
from uabr.eval import prompts as PR
from uabr.eval.batched import greedy_batch
from uabr.eval.generate import load_model
from uabr.eval.metrics import alias_em
from uabr.retrieval.select import select_non_overlapping
from uabr.routing.metrics import auroc, auprc, clustered_bootstrap_auroc
from uabr.routing.pilot import decision_slot, top_fraction, outcome, paired_cluster_bootstrap, decide, flips_by_margin
from uabr.routing.router import router_prompt
from uabr.train.sft import STANDARD, instruction_batches, train_sft, sft_val_loss, count_tokens

## 2 · Configuration and inputs

In [ ]:
import numpy as np, pandas as pd
OUT = "/kaggle/working/07_sft_pilot" if ON_KAGGLE else os.path.abspath("outputs/07_sft_pilot")
os.makedirs(OUT, exist_ok=True)
DEV = "cuda" if torch.cuda.is_available() else "cpu"
if ON_KAGGLE: assert DEV == "cuda", "No GPU - set Accelerator to GPU T4"

SEEDS = [int(s) for s in os.environ.get("SEEDS", "11,29").split(",")]
SAVE_FRACS = [0.25, 0.5]                                  # + final.pt = 100 %
SFT_BUDGET = int(os.environ.get("SFT_BUDGET", 6_000_000)); SFT_VAL = int(os.environ.get("SFT_VAL", 500))
SFT_MAXLEN, SFT_SEED = 512, 2026                          # as uabr-slm-main pack_sft
CFG = dict(STANDARD)                                      # batch 32, LR 5e-5 constant, warm-up 30, (0.9, 0.95), wd 0.01, clip 1, fp16
N_BOOT = int(os.environ.get("N_BOOT", 1000))
BUDGETS = [0.05, 0.10, 0.20]
MAX_NEW, SEQ = 16, 512                                    # as notebook 06
LIMIT = int(os.environ.get("LIMIT", 0))

def one_input(pattern, env=None):
    """Exactly one match required (two attached 06 runs would be ambiguous)."""
    if env and os.environ.get(env): return os.environ[env]
    hits = sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    assert len(hits) == 1, f"{pattern}: expected exactly 1 attached file, found {len(hits)}: {hits}"
    return hits[0]

popqa = pd.read_json(find_input("00_popqa_splits/popqa_prepared.jsonl", "POPQA_PREPARED"), lines=True)
tok = T.load(find_input("01_corpus/tokenizer/tokenizer.json", "TOKENIZER"))
top = pd.read_parquet(find_input("02_bm25/retrieval/popqa_top20.parquet", "TOP20"))
wins = pd.read_parquet(find_input("02_bm25/retrieval/windows_retrieved.parquet", "WINDOWS"))
POLICY_PATH = one_input("06_routing/routing_policy.json", "POLICY")
POLICY = json.load(open(POLICY_PATH)); D06 = os.path.dirname(POLICY_PATH)
assert POLICY["format"] == "action", f"notebook 07 is written for the action router; 06 chose {POLICY['format']}"
ROUTER = os.path.join(D06, "router_action.pt"); assert os.path.exists(ROUTER), ROUTER
LAM = float(POLICY["lambda"])
V = pd.read_parquet(os.path.join(D06, "labels_val.parquet"))
for c in ["y", "router_score", "margin", "em_A", "em_C", "group_id"]:
    assert c in V.columns, f"labels_val.parquet has no column {c} - attach the 06 v2 output"
if LIMIT: V = V.head(LIMIT)
V = V.reset_index(drop=True)
WTEXT = dict(zip(wins["doc_id"], wins["text"]))
WART, WSTART = dict(zip(wins["doc_id"], wins["article_idx"])), dict(zip(wins["doc_id"], wins["start_word"]))
TOPDOCS = dict(zip(top["id"], top["doc_ids"]))
stop_ids = [i for t, i in tok.get_vocab().items() if "\n" in tok.decode([i])]
print(f"theta_route,preSFT = {ROUTER} | lambda {LAM} | validation questions {len(V):,} (CALL labels {V.y.mean():.1%}) | device {DEV}")

## 3 · Instruction data: download (pinned revision), overlap audit, ~6 M-token budget

In [ ]:
SFT_FILE = f"{OUT}/sft_examples.parquet"
if os.environ.get("SFT_LOCAL"):                                   # local tests only
    rows = [json.loads(l) for l in open(os.environ["SFT_LOCAL"])]; REVS = {"local": os.environ["SFT_LOCAL"]}
else:
    from datasets import load_dataset
    from huggingface_hub import HfApi
    REVS = {k: HfApi().dataset_info(v).sha for k, v in SOURCES.items()}
    dolly = load_dataset(SOURCES["dolly15k"], split="train", revision=REVS["dolly15k"])
    alpaca = load_dataset(SOURCES["alpaca-cleaned"], split="train", revision=REVS["alpaca-cleaned"])
    rows = rows_from_hf(dolly, alpaca)
print(f"{len(rows):,} instruction rows | revisions {REVS}")

# overlap audit against every in_main validation + test fact (subject AND a gold alias in the same example)
F = popqa[popqa.split.isin(["val", "test"]) & popqa.in_main]
texts = [f"{r['instruction']}\n{r['context']}\n{r['response']}" for r in rows]
hits = audit_overlap(texts, zip(F.id, F.subj, F.gold_aliases))
fq = F.set_index("id")
aud = pd.DataFrame([{"sft_id": rows[i]["id"], "fact_id": f, "split": fq.loc[f, "split"], "question": fq.loc[f, "question"],
                     "answer": fq.loc[f, "answer"], "sft_text": texts[i][:300]} for i, fs in hits.items() for f in fs])
aud.to_csv(f"{OUT}/sft_overlap_audit.csv", index=False)
n_flag = len(hits)
print(f"overlap audit: {n_flag} instruction rows state {aud.fact_id.nunique() if n_flag else 0} val/test facts "
      f"({(aud.split == 'test').sum() if n_flag else 0} test links) -> removed")
if n_flag: display(aud.head(10))
rows = [r for i, r in enumerate(rows) if i not in hits]

SFT_TRAIN, SFT_VALX, TR_IDS, VA_IDS, SFT_STATS = build_sft(tok, rows, SFT_MAXLEN, SFT_BUDGET, SFT_VAL, SFT_SEED)
SFT_STATS.update(removed_overlap=n_flag, revisions=REVS)
pd.DataFrame({"sft_id": TR_IDS + VA_IDS, "part": ["train"] * len(TR_IDS) + ["val"] * len(VA_IDS),
              "tokens": [t for t, _ in SFT_TRAIN + SFT_VALX],
              "n_prompt": [f.index(True) for _, f in SFT_TRAIN + SFT_VALX]}).to_parquet(SFT_FILE, index=False)
print(json.dumps(SFT_STATS, indent=1))
L = np.array([len(t) for t, _ in SFT_TRAIN]); print(f"example length: median {np.median(L):.0f}, p90 {np.quantile(L, .9):.0f}, max {L.max()}")
ex_t, ex_f = SFT_TRAIN[0]
print("example (target part in [[ ]]):", tok.decode([t for t, f in zip(ex_t, ex_f) if not f])[:200], "[[", tok.decode([t for t, f in zip(ex_t, ex_f) if f])[:120], "]]")
assert T.ANSWER_ID not in ex_t and T.CALL_ID not in ex_t

## 4 · Plain SFT from θ_route,preSFT (seeds 11 and 29; checkpoints at 25 %, 50 %, 100 %)

In [ ]:
CKPTS = [("all", 0.0, ROUTER)]
for seed in SEEDS:
    batches = instruction_batches(SFT_TRAIN, CFG["batch_size"], seed)
    total = len(batches); save_at = sorted({max(1, round(f * total)) for f in SAVE_FRACS})
    run = f"{OUT}/sft_s{seed}"
    prev = sorted(glob.glob(f"/kaggle/input/**/07_sft_pilot/sft_s{seed}/summary.json", recursive=True))
    if prev and json.load(open(prev[0]))["updates"] == total:
        run = os.path.dirname(prev[0]); stage(f"seed {seed}: reusing SFT checkpoints from {run}")
    elif not os.path.exists(f"{run}/final.pt"):
        stage(f"seed {seed}: {total} updates | {count_tokens(SFT_TRAIN)['tokens']:,} instruction tokens | checkpoints at {save_at} + final")
        m = load_model(ROUTER, DEV)
        summ = train_sft(m, batches, CFG, seed, run, save_at_steps=save_at, log=stage)
        stage(f"seed {seed}: done in {summ['minutes']} min"); del m; torch.cuda.empty_cache() if DEV == "cuda" else None
    for s in save_at: CKPTS.append((seed, s / total, f"{run}/step_{s}.pt"))
    CKPTS.append((seed, 1.0, f"{run}/final.pt"))
for c in CKPTS: assert os.path.exists(c[2]), c
print(pd.DataFrame(CKPTS, columns=["seed", "frac", "path"]).to_string(index=False))

## 5 · Evaluate every checkpoint on validation (decision slot, fresh closed/evidence EM, instruction loss)

In [ ]:
RP = [router_prompt(tok, q, "action") for q in V.question]                     # ... <ASSISTANT>
CP = [Q.closed_prompt_ids(tok, q) for q in V.question]                         # ... <ASSISTANT><ANSWER>
EP = []
for r in V.itertuples(index=False):
    sel, _ = select_non_overlapping(TOPDOCS[r.id], WART, WSTART, k=3, window_words=60)
    EP.append(Q.evidence_prompt_ids(tok, r.question, [WTEXT[d] for d in sel], SEQ - MAX_NEW)[0])

def evaluate(path):
    m = load_model(path, DEV)
    d = decision_slot(m, RP, DEV)
    gA = greedy_batch(m, CP, DEV, MAX_NEW, stop_ids=stop_ids); gC = greedy_batch(m, EP, DEV, MAX_NEW, stop_ids=stop_ids)
    d["pred_A"] = [PR.clean_generation(tok.decode(g)) for g in gA]; d["pred_C"] = [PR.clean_generation(tok.decode(g)) for g in gC]
    d["fresh_em_A"] = [alias_em(p, g) for p, g in zip(d.pred_A, V.gold_aliases)]
    d["fresh_em_C"] = [alias_em(p, g) for p, g in zip(d.pred_C, V.gold_aliases)]
    vl = sft_val_loss(m, SFT_VALX, DEV); del m
    if DEV == "cuda": torch.cuda.empty_cache()
    for c in ["id", "group_id", "prop", "pop_bin", "dose_group", "y", "em_A", "em_C"]: d[c] = V[c].values
    return d, vl

PRED, VLOSS = {}, {}
os.makedirs(f"{OUT}/predictions", exist_ok=True)
for seed, frac, path in CKPTS:
    key = (seed, frac); PRED[key], VLOSS[key] = evaluate(path)
    PRED[key].to_parquet(f"{OUT}/predictions/s{seed}_{int(round(frac*100))}pct.parquet", index=False)
    p = PRED[key]
    stage(f"seed {seed} {frac:4.0%}: AUROC {auroc(p.y, p.score):.3f} | default call {np.mean(p.score > 0):.1%} | closed EM {p.fresh_em_A.mean():.1%} "
          f"| evidence EM {p.fresh_em_C.mean():.1%} | P(action) {p.p_action.mean():.3f} | instr val loss {VLOSS[key]:.3f}")
P0 = PRED[("all", 0.0)]
DIFF0 = float(np.abs(P0.score.values - V.router_score.values).max())
THR = {b: float(np.quantile(P0.score, 1 - b)) for b in BUDGETS}               # frozen thresholds of the 0 % point
print(f"0 % point reproduces notebook 06 router scores: max |diff| = {DIFF0:.2e} | frozen thresholds {THR} "
      f"(06 json: {POLICY['thresholds_val']})")

## 6 · Pilot table (0 / 25 / 50 / 100 %)

In [ ]:
def summarize(p, key):
    row = {"seed": key[0], "frac": key[1], "instruction_val_loss": VLOSS[key],
           "auroc": auroc(p.y, p.score), "auprc": auprc(p.y, p.score),
           "auroc_fresh_gain": auroc((p.fresh_em_C > p.fresh_em_A).astype(int), p.score),
           "default_call_rate": float(np.mean(p.score > 0)),
           "closed_em": p.fresh_em_A.mean(), "evidence_em": p.fresh_em_C.mean()}
    for b in BUDGETS:
        k = int(b * 100)
        o = outcome(top_fraction(p.score, b), p.fresh_em_A, p.fresh_em_C)
        row[f"acc_budget_{k}"] = o["accuracy"]; row[f"U_budget_{k}"] = o["accuracy"] - LAM * o["call_rate"]
        f = outcome(p.score.values >= THR[b], p.fresh_em_A, p.fresh_em_C)
        row[f"acc_frozen_thr_{k}"] = f["accuracy"]; row[f"frozen_thr_{k}_call_rate"] = f["call_rate"]
        if k == 10:
            row["harmful_answer_rate_10"] = o["harmful_answer_rate"]
            row["acc_budget_10_theta0_rewards"] = outcome(top_fraction(p.score, b), p.em_A, p.em_C)["accuracy"]   # A3a
    row.update(emission_rate=float(p.argmax_is_action.mean()), p_action_mean=float(p.p_action.mean()),
               action_rank_median=float(p.action_rank.median()))
    return row
TABLE = pd.DataFrame([summarize(PRED[k], k) for k in PRED])
TABLE.to_csv(f"{OUT}/pilot_table.csv", index=False)
pd.set_option("display.width", 250)
display(TABLE[["seed", "frac", "instruction_val_loss", "auroc", "auprc", "acc_budget_10", "acc_frozen_thr_10", "frozen_thr_10_call_rate",
               "default_call_rate", "harmful_answer_rate_10", "closed_em", "evidence_em", "emission_rate", "p_action_mean"]].round(3))

## 7 · Decision: 100 % minus 0 %, paired subject-clustered bootstrap (rule fixed in advance)

In [ ]:
def metric_fn(name, A, B):
    D = {"a": A, "b": B}
    def f(idx, w):
        p = D[w]; s = p.score.values[idx]
        if name == "auroc": return auroc(p.y.values[idx], s)
        if name == "acc_budget_10": return outcome(top_fraction(s, 0.10), p.fresh_em_A.values[idx], p.fresh_em_C.values[idx])["accuracy"]
        if name == "acc_frozen_thr_10": return outcome(s >= THR[0.10], p.fresh_em_A.values[idx], p.fresh_em_C.values[idx])["accuracy"]
    return f
PER_SEED = {}
for seed in SEEDS:
    A = PRED[(seed, 1.0)]
    diffs = {k: paired_cluster_bootstrap(metric_fn(k, A, P0), V.group_id.values, n_boot=N_BOOT, seed=seed)
             for k in ["auroc", "acc_budget_10", "acc_frozen_thr_10"]}
    sig = {k: bool(v["ci_hi"] < 0) for k, v in diffs.items()}
    PER_SEED[str(seed)] = {"diff_100_minus_0": diffs, "significant_drop": sig}
    print(f"seed {seed}: " + " | ".join(f"{k} {v['diff']:+.3f} [{v['ci_lo']:+.3f}, {v['ci_hi']:+.3f}]" for k, v in diffs.items()))
DECISION = decide(PER_SEED)
SFT_LEARNED = bool(all(VLOSS[(s, 1.0)] < VLOSS[("all", 0.0)] for s in SEEDS))
print(f"SFT learned (instruction val loss fell in every seed): {SFT_LEARNED} "
      f"({VLOSS[('all', 0.0)]:.3f} -> " + ", ".join(f"{VLOSS[(s, 1.0)]:.3f}" for s in SEEDS) + ")")
print(f"\nDECISION: {DECISION}")

## 8 · A4 mechanism check: does a small pre-SFT margin predict a flip?

In [ ]:
A4 = {}
for seed in SEEDS:
    for frac in [f for (s, f) in PRED if s == seed]:
        p = PRED[(seed, frac)]
        r0 = flips_by_margin(P0.score, p.score, P0.margin, V.group_id.values)                       # boundary 0 (m_i = |z_C - z_A|)
        t10 = THR[0.10]
        r1 = flips_by_margin(P0.score, p.score, np.abs(P0.score - t10), V.group_id.values, t10, t10)   # frozen 10 % threshold
        A4[(seed, frac)] = (r0, r1)
        a, b = r0["auroc_small_margin_predicts_flip"], r1["auroc_small_margin_predicts_flip"]
        print(f"seed {seed} {frac:4.0%} | boundary 0: flip {r0['flip_rate']:.1%} (A->C {r0['answer_to_call']}, C->A {r0['call_to_answer']}), "
              f"AUROC(-m -> flip) {a['auroc']:.3f} [{a['lo']:.3f}, {a['hi']:.3f}] | frozen 10 %: flip {r1['flip_rate']:.1%} "
              f"(A->C {r1['answer_to_call']}, C->A {r1['call_to_answer']}), AUROC {b['auroc']:.3f} [{b['lo']:.3f}, {b['hi']:.3f}]")
for seed in SEEDS:
    print(f"\nseed {seed}, 100 %: flips by pre-SFT margin quintile (boundary 0)")
    display(A4[(seed, 1.0)][0]["table"].round(3))
A4_SUPPORTED = all(np.nan_to_num(A4[(s, 1.0)][0]["auroc_small_margin_predicts_flip"]["lo"]) > 0.5 for s in SEEDS)
print(f"A4: small margin predicts flips at 100 % in every seed (AUROC CI lower bound > 0.5): {A4_SUPPORTED}"
      + ("" if A4_SUPPORTED else "  -> the boundary term is NOT supported by this pilot; report it"))

## 9 · A5 action-slot collapse, A3 utility, breakdowns

In [ ]:
print("A5: (i) free-generation emission = argmax over the full vocabulary at <ASSISTANT> is <ANSWER> or <CALL>;")
print("    (ii) two-way AUROC from z_CALL - z_ANSWER only = 'auroc'; (iii) re-tuned 10 % threshold = 'acc_budget_10'")
display(TABLE[["seed", "frac", "emission_rate", "p_action_mean", "action_rank_median", "auroc", "acc_budget_10", "default_call_rate"]].round(4))
print("A3: (a) retention vs frozen theta0 labels/rewards  (b) utility with fresh rewards of each checkpoint; lambda =", LAM)
display(TABLE[["seed", "frac", "auroc", "acc_budget_10_theta0_rewards", "U_budget_5", "U_budget_10", "U_budget_20", "closed_em", "evidence_em"]].round(4))
ref = []
for key, p in PRED.items():
    gain = (p.fresh_em_C - p.fresh_em_A).values
    ref.append({"seed": key[0], "frac": key[1], "always_answer": p.fresh_em_A.mean(), "always_call": p.fresh_em_C.mean(),
                **{f"oracle_U_{int(b*100)}": outcome(top_fraction(gain, b), p.fresh_em_A, p.fresh_em_C)["accuracy"] - LAM * b for b in BUDGETS}})
display(pd.DataFrame(ref).round(4))
for key in [("all", 0.0)] + [(s, 1.0) for s in SEEDS]:
    p = PRED[key].assign(call10=lambda d: d.score >= THR[0.10])
    print(f"call rate at the frozen 10 % threshold, seed {key[0]} {key[1]:.0%}:",
          p.groupby("dose_group").call10.mean().round(3).to_dict(), "| by pop_bin", p.groupby("pop_bin").call10.mean().round(3).to_dict())
for seed in SEEDS:
    p = PRED[(seed, 1.0)]
    per_rel = pd.DataFrame({r: {"auroc_0": auroc(g0.y, g0.score), "auroc_100": auroc(g.y, g.score)}
                            for (r, g), (_, g0) in zip(p.groupby("prop"), P0.groupby("prop")) if g.y.nunique() == 2 and len(g) >= 20}).T
    print(f"per-relation AUROC, seed {seed}:"); display(per_rel.round(3))

## 10 · Save + report

In [ ]:
report = {"notebook": "07_sft_pilot", "decision": DECISION, "per_seed": PER_SEED, "sft_learned": SFT_LEARNED,
          "frozen_thresholds_from_0pct": THR, "lambda": LAM, "n_val_questions": len(V), "n_boot": N_BOOT,
          "A4": {f"{k[0]}_{int(round(k[1]*100))}pct": {"boundary0": {x: v for x, v in r0.items() if x != "table"},
                                                       "frozen_thr10": {x: v for x, v in r1.items() if x != "table"}} for k, (r0, r1) in A4.items()},
          "A4_supported": A4_SUPPORTED, "sft_data": SFT_STATS, "sft_config": {k: (list(v) if isinstance(v, tuple) else v) for k, v in CFG.items()},
          "seeds": SEEDS, "checkpoints": [[str(s), f, p] for s, f, p in CKPTS], "router": ROUTER, "routing_policy": POLICY,
          "zero_point_max_score_diff_vs_06": DIFF0, "code": {"repo": REPO_URL, "ref": REPO_REF, "commit": CODE_COMMIT}}
json.dump(report, open(f"{OUT}/pilot_report.json", "w"), indent=1, default=float)
pd.concat([A4[k][0]["table"].assign(seed=k[0], frac=k[1], boundary="0") for k in A4] +
          [A4[k][1]["table"].assign(seed=k[0], frac=k[1], boundary="frozen_thr10") for k in A4]).to_csv(f"{OUT}/a4_flips_by_margin.csv", index=False)
for f in sorted(glob.glob(f"{OUT}/**/*", recursive=True)):
    if os.path.isfile(f): print(f"{os.path.getsize(f)/1e6:9.1f} MB  {os.path.relpath(f, OUT)}")

## 11 · Gate report (validity of the pilot; the decision itself is not a pass/fail)

In [ ]:
checks = []
def add(name, ok, detail, severity="fail"):
    checks.append({"name": name, "passed": bool(ok), "detail": detail, "severity": severity})
add("zero_point_is_06_router", DIFF0 < 1e-2, f"max |score - 06 router_score| = {DIFF0:.2e}")
a0 = TABLE.loc[TABLE.seed == "all", "auroc"].item()
add("zero_point_routes (AUROC >= 0.65, as G6)", a0 >= 0.65, f"0 % AUROC {a0:.3f}")
dc0 = TABLE.loc[TABLE.seed == "all", "default_call_rate"].item()
add("zero_point_default_decision_not_collapsed", 0 < dc0 < 1, f"0 % default call rate (z_CALL > z_ANSWER) {dc0:.1%}", "warn")
add("sft_learned", SFT_LEARNED, "instruction val loss " + ", ".join(f"{k}: {v:.3f}" for k, v in VLOSS.items() if k[1] in (0.0, 1.0)))
add("sft_format_has_no_action_tokens", all(T.ANSWER_ID not in t and T.CALL_ID not in t for t, _ in SFT_TRAIN[:2000]), "plain SFT (A5 primary)")
add("overlap_audit_done", True, f"{n_flag} instruction rows removed (sft_overlap_audit.csv)", "warn")
add("A4_small_margin_predicts_flips", A4_SUPPORTED, "AUROC(-m -> flip) CI lower bound > 0.5 at 100 % in every seed", "warn")

fails = [c for c in checks if not c["passed"] and c["severity"] == "fail"]
warns = [c for c in checks if not c["passed"] and c["severity"] == "warn"]
status = "FAIL" if fails else ("PASS_WITH_WARNINGS" if warns else "PASS")
for c in checks:
    print(f"[{'PASS' if c['passed'] else c['severity'].upper():4}] {c['name']:<38} {c['detail']}")

json.dump({"gate": "phase5_pilot", "status": status, "decision": DECISION, "checks": checks}, open(f"{OUT}/gate_report.json", "w"), indent=2, default=str)
print(f"\nPILOT VALIDITY: {status} | DECISION: {DECISION}")

## Next
- **GO:** Phase 6 — replay arms E2–E7 from the same θ_route,preSFT with identical SFT batches (seeds 11, 29), equal replay
  token budget and reported ANSWER:CALL ratio (A7); the 100 % checkpoints here are arm E1.
- **RECALIBRATION_ONLY:** report threshold recalibration as the simpler fix; Phase 6 results become secondary.
- **NO_CHANGE:** check SFT learned and the 0 % policy is healthy; optionally run the preregistered 200 % duration once;
  if still flat, report the robustness boundary.